# Naive bayes

## Importación de librerías, paquetes y definición de constantes

In [125]:
#import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# Split para modelado
from sklearn.model_selection import train_test_split
# Extract
from sklearn.feature_extraction.text import CountVectorizer
# To save models
import json
import pickle
# Modelado
from sklearn.naive_bayes import MultinomialNB, BernoulliNB, GaussianNB
# Métricas
from sklearn.datasets import make_classification
from utils import get_classifier_metrics
from sklearn.metrics import accuracy_score, classification_report
# Optimizar
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV


## Recopilación de datos

In [85]:
df = pd.read_csv("../data/raw/playstore_reviews.csv")
df.head()

,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0


## Análisis descriptivo 

In [86]:
df

,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0
...,...,...,...
886,com.rovio.angrybirds,loved it i loooooooooooooovvved it because it...,1
887,com.rovio.angrybirds,all time legendary game the birthday party le...,1
888,com.rovio.angrybirds,ads are way to heavy listen to the bad review...,0
889,com.rovio.angrybirds,fun works perfectly well. ads aren't as annoy...,1


In [87]:
df.shape

(891, 3)

In [88]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   package_name  891 non-null    object
 1   review        891 non-null    object
 2   polarity      891 non-null    int64 
dtypes: int64(1), object(2)
memory usage: 21.0+ KB


In [89]:
df.nunique()

package_name     23
review          891
polarity          2
dtype: int64

## Limpieza de datos

### Eliminar duplicados:

In [90]:
df.duplicated().sum()

np.int64(0)

### No hay duplicados pero nos aseguramos de borrarlos

In [91]:
if df.duplicated().sum():
    df = df.drop_duplicates(keep='first')
print(df.shape)
df.head()

(891, 3)


,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0


In [92]:
df.duplicated().sum()

np.int64(0)

### Valores nulos o faltantes:

In [93]:
df.isnull().sum().sort_values(ascending=False) / len(df)

package_name    0.0
review          0.0
polarity        0.0
dtype: float64

#### No hay valores nulos

### Limpieza de datos: Eliminar información irrelevante

#### Borramos la variable 'package_name' porque no influye en la polaridad de los comentarios, solo necesitamos el texto

In [94]:
df.drop(['package_name'], axis = 1, inplace = True)
df

,review,polarity
0,privacy at least put some option appear offli...,0
1,"messenger issues ever since the last update, ...",0
2,profile any time my wife or anybody has more ...,0
3,the new features suck for those of us who don...,0
4,forced reload on uploading pic on replying co...,0
...,...,...
886,loved it i loooooooooooooovvved it because it...,1
887,all time legendary game the birthday party le...,1
888,ads are way to heavy listen to the bad review...,0
889,fun works perfectly well. ads aren't as annoy...,1


## Split de datos


#### Limpiamos la cadena de string para que no tenga espacios y tenga solo minúsculas

In [95]:
df["review"] = df["review"].str.strip().str.lower()

In [96]:
X = df['review']
y = df['polarity']
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=24)
X_train

68     after observation of all above things v should...
768                     errors and bugs app of error....
722    still not cool guys please, this is still not ...
94     facebook ripoff used to see my connections new...
640    too many useless features does anybody actuall...
                             ...                        
145    1 of our 2 favorite games! it is a delight pla...
401    two stars cuz after the update it isnt working...
343    contacts and delays is it just me? with the ne...
192    always fun, but... i like this new frozen shad...
418    always sorry, messenger has stopped. besides, ...
Name: review, Length: 712, dtype: object

#### Vectorizamos la cadena de string

In [97]:
vec_model = CountVectorizer(stop_words = "english")
vec_model


,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,'english'
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'word'


#### `CountVectorizer` necesita que se le pase una cadena de string para poder trabjar

In [98]:
X_train = vec_model.fit_transform(X_train).toarray()
X_test = vec_model.transform(X_test).toarray()


#### `CountVectorizer` toma la frecuencia absoluta de cada token

In [123]:
X_train[0].sum()

np.int64(24)

## Generación del modelo Multinomial

In [127]:
naive_bayes_multi = MultinomialNB()
naive_bayes_multi.fit(X_train,y_train)
y_pred_test_multi = naive_bayes_multi.predict(X_test)


In [128]:
y_pred_train_multi = naive_bayes_multi.predict(X_train)

In [129]:
accuracy_score(y_test, y_pred_test_multi)

0.8491620111731844

In [130]:
print(classification_report(y_test, y_pred_test_multi))

              precision    recall  f1-score   support

           0       0.84      0.95      0.89       114
           1       0.88      0.68      0.77        65

    accuracy                           0.85       179
   macro avg       0.86      0.81      0.83       179
weighted avg       0.85      0.85      0.84       179



### EL modelo es muy bueno detectando opiniones negativas (recall clase 0= 95%) pero falla al detectar las positivas (recall clase 1=68%),esto puede generar una visión pesimista del contenido, afectando a decisiones futuras

In [131]:
print("Naive Bayes Multinomial:")
%timeit naive_bayes_multi.predict(X_test)

Naive Bayes Multinomial:
944 μs ± 84.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## Generación del modelo Bernoulli

In [133]:
naive_bayes_bernoulli = BernoulliNB()
naive_bayes_bernoulli.fit(X_train,y_train)
y_pred_test_bernoulli = naive_bayes_bernoulli.predict(X_test)

In [134]:
y_pred_train_bernoulli = naive_bayes_multi.predict(X_train)

In [135]:
accuracy_score(y_test, y_pred_test_bernoulli)

0.7430167597765364

In [136]:
print(classification_report(y_test, y_pred_test_bernoulli))

              precision    recall  f1-score   support

           0       0.72      0.97      0.83       114
           1       0.88      0.34      0.49        65

    accuracy                           0.74       179
   macro avg       0.80      0.66      0.66       179
weighted avg       0.78      0.74      0.71       179



## Generación del modelo Gaussian

In [139]:
naive_bayes_gauss = GaussianNB()
naive_bayes_gauss.fit(X_train,y_train)
y_pred_test_gauss = naive_bayes_gauss.predict(X_test)

In [140]:
y_pred_train_gauss = naive_bayes_gauss.predict(X_train)

In [141]:
accuracy_score(y_test, y_pred_test_gauss)

0.7597765363128491

In [142]:
print(classification_report(y_test, y_pred_test_gauss))

              precision    recall  f1-score   support

           0       0.77      0.89      0.82       114
           1       0.73      0.54      0.62        65

    accuracy                           0.76       179
   macro avg       0.75      0.71      0.72       179
weighted avg       0.76      0.76      0.75       179



## Optimización del modelo, escogemos el Multinomial porque es el que mejores métricas obtiene

### Parámetros para la cuadrícula

In [108]:
param_grid = {'alpha': [0.01, 0.1, 1, 1.5, 2, 5],
              'force_alpha': [True, False],
              'fit_prior': [True, False]              
    }
grid_search = GridSearchCV(estimator=naive_bayes_multi,
                           param_grid=param_grid,
                           cv=5,
                           refit='accuracy'
                           
)


In [109]:
grid_search.fit(X_train, y_train)

,estimator,MultinomialNB()
,param_grid,"{'alpha': [0.01, 0.1, ...], 'fit_prior': [True, False], 'force_alpha': [True, False]}"
,scoring,None
,n_jobs,None
,refit,'accuracy'
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,alpha,1.5


In [110]:
grid_search.best_params_

{'alpha': 1.5, 'fit_prior': False, 'force_alpha': True}

In [111]:
grid_multi = grid_search.best_estimator_
y_pred_test = grid_multi.predict(X_test)
y_pred_train = grid_multi.predict(X_train)

In [137]:
accuracy_score(y_test, y_pred_test)

0.8603351955307262

In [143]:
print(classification_report(y_test, y_pred_test))

              precision    recall  f1-score   support

           0       0.85      0.95      0.90       114
           1       0.88      0.71      0.79        65

    accuracy                           0.86       179
   macro avg       0.87      0.83      0.84       179
weighted avg       0.86      0.86      0.86       179



## Comparación de los modelos

In [144]:
metricas = {'Gaussian': accuracy_score(y_test, y_pred_test_gauss),
            'Bernoulli': accuracy_score(y_test, y_pred_test_bernoulli),
            'Multinomial': accuracy_score(y_test, y_pred_test_multi),
            'Multinomial Optimizado': accuracy_score(y_test, y_pred_test)
}
metricas

{'Gaussian': 0.7597765363128491,
 'Bernoulli': 0.7430167597765364,
 'Multinomial': 0.8491620111731844,
 'Multinomial Optimizado': 0.8603351955307262}

### Observamos que el modelo Multinomial es el más preciso:

| Modelo                   | Precisión (%) |
|--------------------------|---------------|
| Gaussian                 | 75.98         |
| Bernoulli                | 74.30         |
| Multinomial              | 84.92         |
| Multinomial Optimizado  | 86.03         |

## Conclusiones

>- El mejor resultado lo obtenemos con el modelo multinomial y optimizando podemos conseguir una mejora a considerar, así que elegimos este modelo.


## Guardado de modelos

In [114]:
models = {'naive_bayes_multi' : naive_bayes_multi,
          'naive_bayes_bernoulli' : naive_bayes_bernoulli,
          'naive_bayes_gauss' : naive_bayes_gauss  
          }
with open('C:/Users/Simón/sayons-intro-ml-new/models/modelos-naive-bayes.pkl', 'wb') as file:
    pickle.dump(models, file)